# 🏦 Projet ML — Prédiction d'Approbation de Prêts

**Module :** Machine Learning — Classification  
**Dataset :** Loan Prediction Dataset (614 demandes, 13 variables)  
**Objectif :** Prédire si un prêt sera approuvé (Y) ou refusé (N)

---

## 📋 Table des matières
1. [Exercice 1 — Définition du problème & collecte de données](#ex1)
2. [Exercice 2 — Sélection des caractéristiques](#ex2)
3. [Exercice 3 — Entraînement, évaluation et optimisation](#ex3)
4. [Exercice 4 — Solutions ML pour problèmes spécifiques](#ex4)
5. [Exercice 5 — Stratégie d'évaluation pour 3 types de modèles](#ex5)

## ⚙️ Installation & Imports

In [ ]:
# Installation des bibliothèques nécessaires
!pip install scikit-learn pandas numpy matplotlib seaborn xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn — Prétraitement
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Sklearn — Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from xgboost import XGBClassifier

# Sklearn — Métriques
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, silhouette_score
)

# Style graphique
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
COLORS = ['#185FA5', '#993C1D', '#0F6E56', '#534AB7', '#854F0B']

print('✅ Tous les imports sont prêts !')

---
<a id='ex1'></a>
# 📌 Exercice 1 — Définition du problème & Collecte de données

## 1.1 Énoncé du problème

Une institution financière souhaite **automatiser et optimiser sa décision d'accord ou de refus de prêts**.  
Actuellement, cette décision repose sur une analyse manuelle longue et coûteuse.

**Objectif ML :** Construire un modèle de **classification binaire supervisée** capable de prédire si un demandeur remboursera son prêt (`Loan_Status = Y`) ou fera défaut (`Loan_Status = N`), à partir de ses données personnelles et financières.

**Bénéfices attendus :**
- Réduction du risque de crédit (moins de défauts de paiement)
- Accélération du traitement des dossiers
- Décisions plus objectives et cohérentes
- Réduction des coûts opérationnels

## 1.2 Types de données nécessaires

| Catégorie | Variables | Utilité |
|---|---|---|
| **Informations personnelles** | Genre, Situation maritale, Nombre de dépendants | Profil socio-démographique |
| **Profil professionnel** | Niveau d'éducation, Travailleur indépendant | Stabilité et niveau de revenu |
| **Informations financières** | Revenu demandeur, Revenu co-demandeur, Montant du prêt | Capacité de remboursement |
| **Historique de crédit** | Credit_History (0 ou 1) | Comportement de remboursement passé |
| **Détails du prêt** | Durée du prêt, Zone géographique | Contexte de la demande |

## 1.3 Sources de données

| Source | Données fournies |
|---|---|
| Dossiers internes de la banque | Revenus, historique de remboursement, montants |
| Agences de notation de crédit | Score de crédit, historique d'incidents |
| Déclarations fiscales / fiches de paie | Revenus vérifiés, situation d'emploi |
| Formulaire de demande de prêt | Genre, situation maritale, dépendants |
| Registres fonciers | Zone géographique de la propriété |

---
<a id='ex2'></a>
# 📌 Exercice 2 — Sélection des caractéristiques

In [ ]:
# ── Chargement du dataset ────────────────────────────────────────────────────
# Option A : depuis Google Drive (si uploadé)
# from google.colab import files
# uploaded = files.upload()  # sélectionner le CSV
# df = pd.read_csv(list(uploaded.keys())[0])

# Option B : téléchargement direct depuis Kaggle (URL publique)
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/loan_prediction.csv'
try:
    df = pd.read_csv(url)
    print('✅ Dataset chargé depuis URL')
except:
    # Option C : dataset intégré minimal pour démonstration
    print('⚠️  URL inaccessible — utilisation des données simulées')
    import io
    data = """Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
LP001002,Male,No,0,Graduate,No,5849,0.0,,360.0,1.0,Urban,Y
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y"""
    df = pd.read_csv(io.StringIO(data))

print(f'Shape : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')

In [ ]:
# ── Exploration initiale ─────────────────────────────────────────────────────
print('=== APERÇU DES 5 PREMIÈRES LIGNES ===')
display(df.head())

print('\n=== TYPES ET VALEURS MANQUANTES ===')
info_df = pd.DataFrame({
    'Type': df.dtypes,
    'Valeurs manquantes': df.isnull().sum(),
    '% manquant': (df.isnull().sum() / len(df) * 100).round(2)
})
display(info_df)

print('\n=== DISTRIBUTION DE LA VARIABLE CIBLE ===')
print(df['Loan_Status'].value_counts())
print(f"Ratio déséquilibre : {df['Loan_Status'].value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# ── Visualisation des features par rapport au Loan_Status ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution des variables catégorielles selon le statut du prêt',
             fontsize=14, fontweight='bold', y=1.01)

cat_features = ['Credit_History', 'Education', 'Property_Area',
                'Married', 'Self_Employed', 'Gender']

for ax, feat in zip(axes.flatten(), cat_features):
    ct = pd.crosstab(df[feat], df['Loan_Status'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#D85A30', '#185FA5'],
            edgecolor='white', linewidth=0.5)
    ax.set_title(f'{feat}', fontweight='bold')
    ax.set_ylabel('% dans la catégorie')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(['Refusé (N)', 'Approuvé (Y)'], fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution des variables numériques ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Distribution des variables numériques selon le statut du prêt',
             fontsize=13, fontweight='bold')

num_features = ['ApplicantIncome', 'LoanAmount', 'CoapplicantIncome']
palette = {'Y': '#185FA5', 'N': '#D85A30'}

for ax, feat in zip(axes, num_features):
    for status, color in palette.items():
        subset = df[df['Loan_Status'] == status][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f'{status}', edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Fréquence')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Justification de la sélection des features ──────────────────────────────
feature_justification = {
    'Credit_History':      'TRÈS HAUTE — Indicateur direct du comportement de remboursement passé. '
                           'Les personnes avec Credit_History=1 ont ~80% d\'approbation vs ~8% pour 0.',
    'ApplicantIncome':     'HAUTE — Revenu principal = capacité de remboursement mensuel.',
    'CoapplicantIncome':   'HAUTE — Revenu combiné augmente la solvabilité totale du foyer.',
    'LoanAmount':          'HAUTE — Montant demandé. Un ratio élevé LoanAmount/Revenu = risque accru.',
    'Loan_Amount_Term':    'MOYENNE — La durée affecte le montant des mensualités et le risque total.',
    'Property_Area':       'MOYENNE — Zone Semiurban montre le taux d\'approbation le plus élevé.',
    'Education':           'MOYENNE — Les diplômés ont statistiquement de meilleurs revenus.',
    'Married':             'MOYENNE — Les couples ont souvent deux revenus, réduisant le risque.',
    'Dependents':          'FAIBLE-MOYENNE — Plus de dépendants = charges plus élevées.',
    'Self_Employed':       'FAIBLE — Revenus moins stables mais impact modéré sur la prédiction.',
    'Gender':              'FAIBLE — Peu d\'impact statistique après contrôle des autres variables.',
    'Loan_ID':             'EXCLUE — Identifiant unique, aucune valeur prédictive.',
}

print('=== SÉLECTION ET JUSTIFICATION DES CARACTÉRISTIQUES ===')
print(f'{"Feature":<22} {"Importance":<10} Justification')
print('-' * 90)
for feat, justif in feature_justification.items():
    level = justif.split(' — ')[0]
    desc  = justif.split(' — ')[1] if ' — ' in justif else justif
    print(f'{feat:<22} {level:<10} {desc}')
    print()

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────
df_fe = df.copy()

# 1. Revenu total du foyer
df_fe['Total_Income'] = df_fe['ApplicantIncome'] + df_fe['CoapplicantIncome']

# 2. Ratio dette/revenu (EMI)
df_fe['Loan_Income_Ratio'] = df_fe['LoanAmount'] / (df_fe['Total_Income'] + 1)

# 3. Log-transformation pour réduire la skewness
df_fe['Log_ApplicantIncome']  = np.log1p(df_fe['ApplicantIncome'])
df_fe['Log_LoanAmount']       = np.log1p(df_fe['LoanAmount'])
df_fe['Log_Total_Income']     = np.log1p(df_fe['Total_Income'])

print('✅ Nouvelles features créées :')
print('  - Total_Income       : ApplicantIncome + CoapplicantIncome')
print('  - Loan_Income_Ratio  : LoanAmount / Total_Income  (ratio dette/revenu)')
print('  - Log_ApplicantIncome: log(1 + ApplicantIncome)   (réduit la skewness)')
print('  - Log_LoanAmount     : log(1 + LoanAmount)        (réduit la skewness)')
print('  - Log_Total_Income   : log(1 + Total_Income)      (réduit la skewness)')
display(df_fe[['ApplicantIncome','CoapplicantIncome','Total_Income',
               'LoanAmount','Loan_Income_Ratio','Log_LoanAmount']].head())

---
<a id='ex3'></a>
# 📌 Exercice 3 — Entraînement, Évaluation et Optimisation

In [ ]:
# ── Prétraitement complet ────────────────────────────────────────────────────
df_model = df_fe.copy()

# 1. Supprimer Loan_ID
df_model.drop(columns=['Loan_ID'], inplace=True)

# 2. Imputation des valeurs manquantes
#    — Variables numériques : médiane
num_cols = ['LoanAmount', 'Loan_Amount_Term', 'Credit_History',
            'Log_LoanAmount', 'Loan_Income_Ratio', 'Log_Total_Income']
for col in num_cols:
    if col in df_model.columns:
        df_model[col].fillna(df_model[col].median(), inplace=True)

#    — Variables catégorielles : mode
cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed']
for col in cat_cols:
    df_model[col].fillna(df_model[col].mode()[0], inplace=True)

print(f'Valeurs manquantes restantes : {df_model.isnull().sum().sum()}')

# 3. Encodage des variables catégorielles
le = LabelEncoder()
encode_cols = ['Gender', 'Married', 'Dependents', 'Education',
               'Self_Employed', 'Property_Area', 'Loan_Status']
for col in encode_cols:
    df_model[col] = le.fit_transform(df_model[col])

print('✅ Encodage terminé')
display(df_model.head(3))

# 4. Séparation X / y
X = df_model.drop(columns=['Loan_Status'])
y = df_model['Loan_Status']

# 5. Split 80% train — 20% test (stratifié)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Standardisation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\n✅ Train : {X_train.shape} | Test : {X_test.shape}')
print(f'Distribution train — Y:{sum(y_train==1)} | N:{sum(y_train==0)}')
print(f'Distribution test  — Y:{sum(y_test==1)}  | N:{sum(y_test==0)}')

In [ ]:
# ── Entraînement de 4 modèles ────────────────────────────────────────────────
models = {
    'Régression Logistique': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42),
    'Arbre de Décision':      DecisionTreeClassifier(max_depth=5, random_state=42),
    'XGBoost':                XGBClassifier(n_estimators=100, random_state=42,
                                             eval_metric='logloss', verbosity=0),
}

results = {}

for name, model in models.items():
    # Utiliser données standardisées pour LR, données brutes pour les arbres
    Xtr = X_train_sc if 'Régression' in name else X_train
    Xte = X_test_sc  if 'Régression' in name else X_test

    model.fit(Xtr, y_train)
    y_pred      = model.predict(Xte)
    y_proba     = model.predict_proba(Xte)[:,1]

    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_proba),
    }
    print(f'✅ {name} entraîné')

print('\n=== COMPARAISON DES PERFORMANCES ===')
summary = pd.DataFrame({
    name: {k: v for k, v in res.items()
           if k in ['Accuracy','Precision','Recall','F1','AUC-ROC']}
    for name, res in results.items()
}).T.round(4)
display(summary.style.highlight_max(axis=0, color='#C6EFCE')
                      .highlight_min(axis=0, color='#FFCCCC'))

In [ ]:
# ── Matrices de confusion ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Matrices de Confusion — Tous les Modèles',
             fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                cmap='Blues', linewidths=0.5, linecolor='white',
                xticklabels=['Refusé (N)', 'Approuvé (Y)'],
                yticklabels=['Refusé (N)', 'Approuvé (Y)'])
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.set_ylabel('Réel')
    ax.set_xlabel('Prédit')

plt.tight_layout()
plt.show()

In [ ]:
# ── Courbes ROC ──────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 6))
plot_colors = ['#185FA5', '#993C1D', '#0F6E56', '#534AB7']

for (name, res), color in zip(results.items(), plot_colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    auc = res['AUC-ROC']
    plt.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0,1],[0,1], 'k--', lw=1.2, label='Aléatoire (AUC = 0.500)')
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR / Rappel)', fontsize=12)
plt.title('Courbes ROC — Comparaison des modèles', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.fill_between([0,1],[0,1], alpha=0.05, color='gray')
plt.tight_layout()
plt.show()

In [ ]:
# ── Importance des features (Random Forest) ──────────────────────────────────
rf_model = results['Random Forest']['model']
importance_df = pd.DataFrame({
    'Feature':    X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
               color='#185FA5', edgecolor='white', linewidth=0.5)

# Colorer les 3 plus importantes en vert
top3_idx = importance_df.index[-3:]
for i, bar in enumerate(bars):
    if importance_df.iloc[i].name in top3_idx:
        bar.set_color('#0F6E56')

ax.set_xlabel('Importance (Gini)', fontsize=12)
ax.set_title('Importance des Features — Random Forest', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, importance_df['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Validation croisée (5-fold stratifiée) ───────────────────────────────────
print('=== VALIDATION CROISÉE 5-FOLD STRATIFIÉE ===')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, model in models.items():
    Xdata = X_train_sc if 'Régression' in name else X_train
    scores_f1  = cross_val_score(model, Xdata, y_train, cv=cv, scoring='f1')
    scores_roc = cross_val_score(model, Xdata, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = {
        'F1 moyen':    scores_f1.mean(),
        'F1 std':      scores_f1.std(),
        'AUC moyen':   scores_roc.mean(),
        'AUC std':     scores_roc.std(),
    }
    print(f'{name:25} | F1: {scores_f1.mean():.4f} ± {scores_f1.std():.4f} '
          f'| AUC: {scores_roc.mean():.4f} ± {scores_roc.std():.4f}')

print('\n💡 Le modèle avec le meilleur F1 moyen ET faible variance est le plus robuste.')

In [ ]:
# ── Rapport final du meilleur modèle ─────────────────────────────────────────
best_name = max(results, key=lambda x: results[x]['F1'])
best_res  = results[best_name]

print(f'=== MEILLEUR MODÈLE : {best_name} ===')
print(f'F1-Score  : {best_res["F1"]:.4f}')
print(f'AUC-ROC   : {best_res["AUC-ROC"]:.4f}')
print(f'Rappel    : {best_res["Recall"]:.4f}  ← minimiser les fraudes non détectées')
print(f'Précision : {best_res["Precision"]:.4f}')
print()
print('=== RAPPORT DE CLASSIFICATION DÉTAILLÉ ===')
print(classification_report(y_test, best_res['y_pred'],
                             target_names=['Refusé (N)', 'Approuvé (Y)']))

---
<a id='ex4'></a>
# 📌 Exercice 4 — Solutions ML pour problèmes spécifiques

## Scénario 1 — Prévision des cours boursiers 📈

**Type d'apprentissage : Supervisé — Régression**

Les cours boursiers sont des **valeurs continues** à prédire à partir de données historiques (cours passés, volumes, indicateurs techniques). C'est un problème de **régression supervisée** car :
- On dispose de données historiques étiquetées (prix passés → prix futur connu)
- La sortie est une valeur numérique continue (ex : prix dans 7 jours)

**Algorithmes recommandés :** LSTM (Long Short-Term Memory), ARIMA, Random Forest Regressor, XGBoost Regressor.

**Métriques d'évaluation :** MAE (Mean Absolute Error), RMSE, R²

---

## Scénario 2 — Organiser une bibliothèque 📚

**Type d'apprentissage : Non supervisé — Clustering**

On n'a pas de catégories prédéfinies : on veut **découvrir des groupes naturels** de livres similaires par leur contenu, thèmes ou genre littéraire. C'est non supervisé car :
- Aucune étiquette de catégorie prédéfinie n'existe
- Le but est de découvrir la structure cachée dans les données

**Algorithmes recommandés :** K-Means (clustering par similarité), LDA (Latent Dirichlet Allocation pour les textes), DBSCAN.

**Métriques d'évaluation :** Score de silhouette, méthode du coude (Elbow).

---

## Scénario 3 — Robot dans un labyrinthe 🤖

**Type d'apprentissage : Apprentissage par renforcement (Reinforcement Learning)**

Le robot apprend par **essais-erreurs** en interagissant avec son environnement :
- Récompense positive quand il avance vers la sortie
- Récompense négative quand il heurte un mur ou recule
- Pas de données étiquetées — juste un environnement et des récompenses

**Algorithmes recommandés :** Q-Learning, Deep Q-Network (DQN), A* (pour labyrinthe statique).

**Métriques :** Récompense cumulative, nombre de pas pour atteindre la sortie, convergence de la politique.

In [ ]:
# ── Visualisation des 3 scénarios ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exercice 4 — Types de ML par scénario',
             fontsize=14, fontweight='bold')

# Scénario 1 : Régression — simulation de cours boursier
ax = axes[0]
np.random.seed(42)
t = np.arange(100)
price = 100 + np.cumsum(np.random.randn(100) * 2)
pred_t = np.arange(95, 110)
pred_p = price[94] + np.cumsum(np.random.randn(15) * 1.5)
ax.plot(t, price, color='#185FA5', lw=2, label='Historique')
ax.plot(pred_t, pred_p, color='#D85A30', lw=2, linestyle='--', label='Prédiction')
ax.axvline(x=99, color='gray', linestyle=':', lw=1)
ax.set_title('Cours boursier\n(Supervisé — Régression)', fontweight='bold')
ax.set_xlabel('Temps'); ax.set_ylabel('Prix')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Scénario 2 : Clustering
ax = axes[1]
np.random.seed(0)
centers = [(1,1), (4,4), (7,1)]
genres  = ['Science-Fiction', 'Romance', 'Policier']
colors_cl = ['#185FA5', '#0F6E56', '#993C1D']
for (cx,cy), genre, color in zip(centers, genres, colors_cl):
    pts = np.random.randn(30, 2) * 0.8 + [cx, cy]
    ax.scatter(pts[:,0], pts[:,1], c=color, alpha=0.7, s=40,
               edgecolors='white', lw=0.3, label=genre)
    ax.scatter(cx, cy, c=color, s=200, marker='*', zorder=5, edgecolors='black')
ax.set_title('Bibliothèque\n(Non supervisé — Clustering)', fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Scénario 3 : Q-learning — récompenses cumulatives
ax = axes[2]
episodes = np.arange(1, 201)
# Courbe de convergence type Q-learning
rewards = -50 * np.exp(-episodes/60) + 45 + np.random.randn(200) * 3
ax.plot(episodes, rewards, color='#534AB7', lw=1.5, alpha=0.7)
# Moyenne mobile
window = pd.Series(rewards).rolling(20).mean()
ax.plot(episodes, window, color='#D85A30', lw=2.5, label='Tendance (moy. mobile 20)')
ax.axhline(y=40, color='gray', linestyle='--', lw=1, label='Récompense optimale')
ax.set_title('Robot labyrinthe\n(Renforcement — Q-Learning)', fontweight='bold')
ax.set_xlabel('Épisodes'); ax.set_ylabel('Récompense cumulative')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
<a id='ex5'></a>
# 📌 Exercice 5 — Stratégie d'évaluation pour 3 types de modèles

In [ ]:
# ── Modèle 1 : Supervisé — Random Forest (classification) ───────────────────
print('='*60)
print('MODÈLE 1 — SUPERVISÉ : Random Forest (Classification)')
print('='*60)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

print('\n--- Métriques d\'évaluation ---')
print(f'Accuracy  : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_rf):.4f}  → parmi les prêts accordés, combien étaient justifiés ?')
print(f'Recall    : {recall_score(y_test, y_pred_rf):.4f}  → parmi tous les bons payeurs, combien détectés ?')
print(f'F1-Score  : {f1_score(y_test, y_pred_rf):.4f}  → équilibre précision/rappel')
print(f'AUC-ROC   : {roc_auc_score(y_test, y_proba_rf):.4f}  → performance globale')

print('\n--- Validation croisée 5-fold ---')
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='f1')
print(f'F1 par fold : {[round(s,4) for s in cv_scores]}')
print(f'F1 moyen    : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n--- Difficultés et limites ---')
print('⚠ Données déséquilibrées (69% Y) : accuracy trompeuse → toujours préférer F1 et Recall')
print('⚠ Surapprentissage possible si arbres trop profonds → utiliser max_depth ou cross-validation')
print('⚠ Interprétabilité limitée → utiliser SHAP values pour expliquer les décisions')

In [ ]:
# ── Modèle 2 : Non supervisé — K-Means (clustering) ─────────────────────────
print('='*60)
print('MODÈLE 2 — NON SUPERVISÉ : K-Means Clustering')
print('='*60)

# Utiliser features numériques pour le clustering
X_cluster = df_model[['ApplicantIncome', 'LoanAmount',
                       'Credit_History', 'Loan_Income_Ratio',
                       'Total_Income']].copy()
X_cluster.fillna(X_cluster.median(), inplace=True)
X_cluster_sc = StandardScaler().fit_transform(X_cluster)

# Méthode du coude
inertia = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster_sc)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_sc, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Elbow
axes[0].plot(K_range, inertia, 'o-', color='#185FA5', lw=2)
axes[0].set_xlabel('Nombre de clusters K')
axes[0].set_ylabel('Inertie (Within-cluster SS)')
axes[0].set_title('Méthode du Coude (Elbow Method)', fontweight='bold')
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K optimal ≈ 3')
axes[0].legend()
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Silhouette
axes[1].bar(K_range, sil_scores, color='#0F6E56', edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Nombre de clusters K')
axes[1].set_ylabel('Score de Silhouette')
axes[1].set_title('Score de Silhouette par K', fontweight='bold')
best_k = list(K_range)[sil_scores.index(max(sil_scores))]
axes[1].axvline(x=best_k, color='red', linestyle='--', alpha=0.7,
                label=f'Meilleur K = {best_k}')
axes[1].legend()
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'\nMeilleur score de silhouette : {max(sil_scores):.4f} pour K={best_k}')
print('Score silhouette proche de 1 = clusters bien séparés et cohérents')
print('\n--- Difficultés et limites ---')
print('⚠ Pas de ground truth → évaluation subjective, pas de "bonne réponse"')
print('⚠ K-Means sensible aux outliers et à l\'initialisation')
print('⚠ Suppose des clusters sphériques — inadapté aux formes irrégulières')

In [ ]:
# ── Modèle 3 : Apprentissage par renforcement — Q-Learning simulé ────────────
print('='*60)
print('MODÈLE 3 — RENFORCEMENT : Q-Learning (Labyrinthe)')
print('='*60)

# Simulation d'un apprentissage Q-Learning sur 300 épisodes
np.random.seed(42)
n_episodes = 300

# Simulation réaliste d'une courbe de convergence Q-learning
epsilon_start, epsilon_end = 1.0, 0.05
epsilons   = np.linspace(epsilon_start, epsilon_end, n_episodes)
# Récompenses : commence négatives (exploration), monte avec l'apprentissage
rewards    = (-80 * np.exp(-np.arange(n_episodes)/80) + 70
              + np.random.randn(n_episodes) * 8)
# Nombre de pas pour atteindre la sortie (décroît avec l'apprentissage)
steps      = (200 * np.exp(-np.arange(n_episodes)/100) + 15
              + np.random.randint(0, 10, n_episodes)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Évaluation du modèle par renforcement (Q-Learning simulé)',
             fontsize=13, fontweight='bold')

# 1. Récompense cumulative
ax = axes[0]
ax.plot(rewards, alpha=0.4, color='#534AB7', lw=1)
roll_r = pd.Series(rewards).rolling(20).mean()
ax.plot(roll_r, color='#534AB7', lw=2.5, label='Moy. mobile 20')
ax.axhline(y=60, color='#0F6E56', linestyle='--', lw=1.5, label='Convergence cible')
ax.set_xlabel('Épisodes'); ax.set_ylabel('Récompense')
ax.set_title('Récompense cumulative\n(doit augmenter)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 2. Nombre de pas (convergence)
ax = axes[1]
ax.plot(steps, alpha=0.4, color='#993C1D', lw=1)
roll_s = pd.Series(steps.astype(float)).rolling(20).mean()
ax.plot(roll_s, color='#993C1D', lw=2.5, label='Moy. mobile 20')
ax.axhline(y=20, color='#0F6E56', linestyle='--', lw=1.5, label='Chemin optimal')
ax.set_xlabel('Épisodes'); ax.set_ylabel('Nombre de pas')
ax.set_title('Convergence\n(nb de pas doit diminuer)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 3. Exploration vs Exploitation (epsilon decay)
ax = axes[2]
ax.fill_between(range(n_episodes), epsilons, alpha=0.3, color='#D85A30', label='Exploration (ε)')
ax.fill_between(range(n_episodes), 1-epsilons, alpha=0.3, color='#185FA5', label='Exploitation (1-ε)')
ax.plot(epsilons, color='#D85A30', lw=2)
ax.set_xlabel('Épisodes'); ax.set_ylabel('Probabilité')
ax.set_title('Exploration vs Exploitation\n(epsilon-greedy decay)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Récompense finale (moy. 20 derniers épisodes) : {rewards[-20:].mean():.2f}')
print(f'Nb de pas final (moy. 20 derniers épisodes)   : {steps[-20:].mean():.1f}')
print(f'Epsilon final (exploitation)                  : {epsilons[-1]:.3f}')
print('\n--- Difficultés et limites ---')
print('⚠ Très coûteux en calcul — des milliers d\'épisodes nécessaires')
print('⚠ Choix de la fonction de récompense critique — mal définie = mauvais comportement')
print('⚠ Difficile à évaluer avant convergence complète')
print('⚠ Pas de garantie de trouver la solution optimale globale')

---
# ✅ Conclusion générale

| Exercice | Résultat clé |
|---|---|
| **Ex1** | Problème de classification binaire déséquilibrée (69% Y / 31% N) |
| **Ex2** | `Credit_History` est la feature la plus prédictive ; `Total_Income` et `Loan_Income_Ratio` apportent de la valeur |
| **Ex3** | Random Forest et XGBoost obtiennent les meilleurs F1 et AUC-ROC ; privilégier le Rappel pour limiter les défauts non détectés |
| **Ex4** | Supervisé (régression) pour bourse ; Non supervisé (clustering) pour bibliothèque ; Renforcement pour robot |
| **Ex5** | F1+AUC pour supervisé ; Silhouette+Elbow pour clustering ; Récompense cumulée+convergence pour RL |

> **Note :** Ne jamais utiliser l'exactitude (accuracy) seule sur des données déséquilibrées.  
> Toujours privilégier **F1-Score, Rappel et AUC-ROC** pour les problèmes de prédiction de défaut.